In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/appledisease/Test/Blotch_Apple/149.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/143.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/131.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/130.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/120.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/124.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/123.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/137.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/127.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/140.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/145.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/135.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/121.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/126.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/142.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/148.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/138.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/146.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/1

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image

In [4]:
train_dir = "/kaggle/input/appledisease/Train"
test_dir = "/kaggle/input/appledisease/Test"

In [5]:
img_size = (128, 128)
batch_size = 32
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=25,
    zoom_range=0.3,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

In [6]:
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

In [7]:
test_datagen = ImageDataGenerator(rescale=1./255)

In [8]:
train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

Found 307 images belonging to 4 classes.


In [9]:
val_gen = val_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)


Found 75 images belonging to 4 classes.


In [10]:
test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

Found 120 images belonging to 4 classes.


In [11]:
for subdir, _, files in os.walk(train_dir):
    for file in files:
        try:
            img_path = os.path.join(subdir, file)
            img = Image.open(img_path)
            img.verify()
        except Exception as e:
            print(f"Corrupted or unreadable file: {img_path} — {e}")

In [12]:
labels_from_gen = train_gen.classes
class_weight = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels_from_gen),
    y=labels_from_gen
)
class_weight = dict(enumerate(class_weight))

In [13]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
lr_reduce = ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.3, verbose=1)

In [14]:
base_model = MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
base_model.trainable = False

I0000 00:00:1764277888.094438      37 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [15]:
model = Sequential()
model.add(base_model)
model.add(GlobalAveragePooling2D())
model.add(Dropout(0.3))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(train_gen.num_classes, activation='softmax'))

In [16]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [17]:
print("Phase 1: Training with base frozen...")
model.fit(train_gen, validation_data=val_gen,
          epochs=10, class_weight=class_weight,
          callbacks=[early_stop, lr_reduce])

Phase 1: Training with base frozen...


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1764277903.587184     117 service.cc:148] XLA service 0x7eea30041990 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1764277903.587924     117 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1764277904.446587     117 cuda_dnn.cc:529] Loaded cuDNN version 90300


 1/10 ━━━━━━━━━━━━━━━━━━━━ 1:41 11s/step - accuracy: 0.2500 - loss: 1.7421

I0000 00:00:1764277908.288788     117 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


10/10 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 0.4039 - loss: 1.5342 - val_accuracy: 0.7067 - val_loss: 0.7086 - learning_rate: 0.0010
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 249ms/step - accuracy: 0.6517 - loss: 0.8063 - val_accuracy: 0.8933 - val_loss: 0.4041 - learning_rate: 0.0010
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 245ms/step - accuracy: 0.7625 - loss: 0.6396 - val_accuracy: 0.9067 - val_loss: 0.3908 - learning_rate: 0.0010
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 254ms/step - accuracy: 0.7633 - loss: 0.6526 - val_accuracy: 0.9200 - val_loss: 0.3465 - learning_rate: 0.0010
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 252ms/step - accuracy: 0.8171 - loss: 0.4745 - val_accuracy: 0.9067 - val_loss: 0.3347 - learning_rate: 0.0010
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 243ms/step - accuracy: 0.8057 - loss: 0.5003 - val_accuracy: 0.8933 - val_loss: 0.3204 - learning_rate: 0.0010
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 236ms/step - accuracy: 0.8493 - loss: 0.4047 - val_accuracy:

In [18]:
print("Phase 2: Fine-tuning...")
base_model.trainable = True

Phase 2: Fine-tuning...


In [19]:
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

In [20]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [21]:
training = model.fit(train_gen, validation_data=val_gen,
          epochs=10, class_weight=class_weight,
          callbacks=[early_stop, lr_reduce])

Epoch 1/10


E0000 00:00:1764278017.745965     120 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1764278017.942968     120 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


 3/10 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - accuracy: 0.6875 - loss: 0.8114

E0000 00:00:1764278025.903992     119 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1764278026.103162     119 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


10/10 ━━━━━━━━━━━━━━━━━━━━ 39s 2s/step - accuracy: 0.7051 - loss: 0.7753 - val_accuracy: 0.8667 - val_loss: 0.3979 - learning_rate: 1.0000e-04
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 237ms/step - accuracy: 0.8479 - loss: 0.4312 - val_accuracy: 0.7733 - val_loss: 0.7446 - learning_rate: 1.0000e-04
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 215ms/step - accuracy: 0.8506 - loss: 0.3368
Epoch 3: ReduceLROnPlateau reducing learning rate to 2.9999999242136255e-05.
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 238ms/step - accuracy: 0.8530 - loss: 0.3344 - val_accuracy: 0.7067 - val_loss: 0.8808 - learning_rate: 1.0000e-04
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 217ms/step - accuracy: 0.9139 - loss: 0.2614 - val_accuracy: 0.7067 - val_loss: 0.8802 - learning_rate: 3.0000e-05
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - accuracy: 0.8338 - loss: 0.3548
Epoch 5: ReduceLROnPlateau reducing learning rate to 8.999999772640877e-06.
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 220ms/step - accuracy: 0.8373 - loss: 0.3507 

In [22]:
best_train_acc = max(training.history['accuracy'])
best_val_acc = max(training.history['val_accuracy'])

In [23]:
print(f"Best Training Accuracy: {best_train_acc:.4f}")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")

Best Training Accuracy: 0.9023
Best Validation Accuracy: 0.8667


In [24]:
print("Best Validation Accuracy:", max(training.history['val_accuracy']))


Best Validation Accuracy: 0.8666666746139526
